In [2]:
#source https://towardsdatascience.com/how-to-create-a-chatbot-with-python-deep-learning-in-less-than-an-hour-56a063bdfc44

import nltk
nltk.download('punkt')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
import json
import pickle

import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
from tensorflow.keras.optimizers import SGD
import random

[nltk_data] Downloading package punkt to /Users/I.Mack/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/I.Mack/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
words = []
classes = []
documents = []
ignore_words = ['?', '!']
data_file = open('intents.json').read()
intents = json.loads(data_file)

In [4]:
for intent in intents['intents']:
    for pattern in intent['patterns']:
        
        w = nltk.word_tokenize(pattern)
        words.extend(w)
        
        documents.append((w, intent['tag']))
        
        if intent['tag'] not in classes:
            classes.append(intent['tag'])

In [5]:
words = [lemmatizer.lemmatize(w.lower()) for w in words if w not in ignore_words]
words = sorted(list(set(classes)))

print(len(documents), 'documents')

print(len(classes), 'classes', classes)

print(len(words), 'unique lemmatized words', words)

pickle.dump(words,open('words.pkl','wb'))
pickle.dump(classes,open('classes.pkl','wb'))

47 documents
9 classes ['greeting', 'goodbye', 'thanks', 'options', 'adverse_drug', 'blood_pressure', 'blood_pressure_search', 'pharmacy_search', 'hospital_search']
9 unique lemmatized words ['adverse_drug', 'blood_pressure', 'blood_pressure_search', 'goodbye', 'greeting', 'hospital_search', 'options', 'pharmacy_search', 'thanks']


In [6]:
training = []
output_empty = [0] * len(classes)
for doc in documents:
    bag = []
    pattern_words = doc[0]
    pattern_words = [lemmatizer.lemmatize(word.lower()) for word in pattern_words]
    for w in words:
        bag.append(1) if w in pattern_words else bag.append(0)
    output_row = list(output_empty)
    output_row[classes.index(doc[1])] = 1
    training.append([bag, output_row])
random.shuffle(training)
training = np.array(training)
train_x = list(training[:,0])
train_y = list(training[:,1])
print('Training data created')
        
    


Training data created


In [10]:
###Keras
model = Sequential()
model.add(Dense(128, input_shape=(len(train_x[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation='softmax'))

sgd = SGD(lr=0.01, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])

hist = model.fit(np.array(train_x), np.array(train_y), epochs=200, batch_size=5, verbose=1)
model.save('chatbot_model.h5', hist)

print('model created')

/Users/I.Mack/opt/anaconda3/lib/python3.7/site-packages/keras/optimizer_v2/optimizer_v2.py:356: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  "The `lr` argument is deprecated, use `learning_rate` instead.")


Epoch 1/200
10/10 [==============================] - 1s 2ms/step - loss: 2.2030 - accuracy: 0.0638
Epoch 2/200
10/10 [==============================] - 0s 2ms/step - loss: 2.1917 - accuracy: 0.1489
Epoch 3/200
10/10 [==============================] - 0s 2ms/step - loss: 2.1967 - accuracy: 0.1489
Epoch 4/200
10/10 [==============================] - 0s 4ms/step - loss: 2.1892 - accuracy: 0.1702
Epoch 5/200
10/10 [==============================] - 0s 3ms/step - loss: 2.1820 - accuracy: 0.1915
Epoch 6/200
10/10 [==============================] - 0s 2ms/step - loss: 2.1813 - accuracy: 0.1489
Epoch 7/200
10/10 [==============================] - 0s 3ms/step - loss: 2.1807 - accuracy: 0.1915
Epoch 8/200
10/10 [==============================] - 0s 2ms/step - loss: 2.1691 - accuracy: 0.1702
Epoch 9/200
10/10 [==============================] - 0s 3ms/step - loss: 2.1327 - accuracy: 0.2128
Epoch 10/200
10/10 [==============================] - 0s 3ms/step - loss: 2.1623 - accuracy: 0.1915
Epoch 11/

10/10 [==============================] - 0s 2ms/step - loss: 2.0024 - accuracy: 0.2553
Epoch 84/200
10/10 [==============================] - 0s 2ms/step - loss: 1.9781 - accuracy: 0.2340
Epoch 85/200
10/10 [==============================] - 0s 2ms/step - loss: 1.9788 - accuracy: 0.2340
Epoch 86/200
10/10 [==============================] - 0s 2ms/step - loss: 2.0011 - accuracy: 0.1915
Epoch 87/200
10/10 [==============================] - 0s 2ms/step - loss: 1.9963 - accuracy: 0.2128
Epoch 88/200
10/10 [==============================] - 0s 2ms/step - loss: 2.0023 - accuracy: 0.2766
Epoch 89/200
10/10 [==============================] - 0s 2ms/step - loss: 1.9795 - accuracy: 0.2128
Epoch 90/200
10/10 [==============================] - 0s 2ms/step - loss: 2.0335 - accuracy: 0.2128
Epoch 91/200
10/10 [==============================] - 0s 2ms/step - loss: 1.9739 - accuracy: 0.2340
Epoch 92/200
10/10 [==============================] - 0s 2ms/step - loss: 2.0225 - accuracy: 0.2128
Epoch 93/200


10/10 [==============================] - 0s 4ms/step - loss: 1.9865 - accuracy: 0.2340
Epoch 165/200
10/10 [==============================] - 0s 3ms/step - loss: 1.9818 - accuracy: 0.2340
Epoch 166/200
10/10 [==============================] - 0s 2ms/step - loss: 1.9799 - accuracy: 0.2340
Epoch 167/200
10/10 [==============================] - 0s 3ms/step - loss: 1.9797 - accuracy: 0.2340
Epoch 168/200
10/10 [==============================] - 0s 3ms/step - loss: 2.0009 - accuracy: 0.2340
Epoch 169/200
10/10 [==============================] - 0s 3ms/step - loss: 1.9763 - accuracy: 0.2340
Epoch 170/200
10/10 [==============================] - 0s 3ms/step - loss: 1.9859 - accuracy: 0.2340
Epoch 171/200
10/10 [==============================] - 0s 2ms/step - loss: 1.9751 - accuracy: 0.2340
Epoch 172/200
10/10 [==============================] - 0s 2ms/step - loss: 1.9860 - accuracy: 0.2340
Epoch 173/200
10/10 [==============================] - 0s 2ms/step - loss: 1.9817 - accuracy: 0.2340
Epoc

In [11]:
###GUI Definitisons

from keras.models import load_model
model = load_model('chatbot_model.h5')
import json
import random
intents = json.loads(open('intents.json').read())
words = pickle.load(open('words.pkl','rb'))
classes = pickle.load(open('classes.pkl','rb'))

In [12]:
def clean_up_sentence(sentence):
    sentence_words = nltk.word_tokenize(sentence)
    sentence_words = [lemmatizer.lemmatize(word.lower()) for word in sentence_words]
    return sentence_words


In [15]:
def bow(sentence, words, show_details=True):
    sentence_words = clean_up_sentence(sentence)
    bag = [0]*len(words)
    for s in sentence_words:
        for i,w in enumerate(words):
            if w == s:
                bag[i] = 1
                if show_details:
                    print('found in bag: %s'%w)
    return(np.array(bag))

In [17]:
def predict_class(sentence, model):
    p = bow(sentence, words, show_details=False)
    res = model.predict(np.array([p]))[0]
    ERROR_THRESHOLD = 0.25
    results = [[i,r] for i,r in enumerate(res) if r>ERROR_THRESHOLD]
    results.sort(key=lambda x: x[1], reverse=True)
    return_list = []
    for r in results:
        return_list.append({'intent': classes[r[0]], 'probability': str(r[1])})
    return return_list

In [18]:
def getResponse(ints, itents_json):
    tag = ints[0]['intent']
    list_of_intents = intents_json['intents']
    for i in list_of_intents:
        if(i['tag']==tag):
            result = random.choice(i['responses'])
            break
        return result

In [19]:
def chatbot_respnse(msg):
    ints = predict_class(msg, model)
    res = getResponse(ints, intents)
    return res

In [20]:
import tkinter
from tkinter import *


def send():
    msg = EntryBox.get("1.0",'end-1c').strip()
    EntryBox.delete("0.0",END)

    if msg != '':
        ChatLog.config(state=NORMAL)
        ChatLog.insert(END, "You: " + msg + '\n\n')
        ChatLog.config(foreground="#442265", font=("Verdana", 12 ))

        res = chatbot_response(msg)
        ChatLog.insert(END, "Bot: " + res + '\n\n')

        ChatLog.config(state=DISABLED)
        ChatLog.yview(END)


base = Tk()
base.title("Hello")
base.geometry("400x500")
base.resizable(width=FALSE, height=FALSE)

#Create Chat window
ChatLog = Text(base, bd=0, bg="white", height="8", width="50", font="Arial",)

ChatLog.config(state=DISABLED)

#Bind scrollbar to Chat window
scrollbar = Scrollbar(base, command=ChatLog.yview, cursor="heart")
ChatLog['yscrollcommand'] = scrollbar.set

#Create Button to send message
SendButton = Button(base, font=("Verdana",12,'bold'), text="Send", width="12", height=5,
                    bd=0, bg="#32de97", activebackground="#3c9d9b",fg='#ffffff',
                    command= send )

#Create the box to enter message
EntryBox = Text(base, bd=0, bg="white",width="29", height="5", font="Arial")
#EntryBox.bind("<Return>", send)


#Place all components on the screen
scrollbar.place(x=376,y=6, height=386)
ChatLog.place(x=6,y=6, height=386, width=370)
EntryBox.place(x=128, y=401, height=90, width=265)
SendButton.place(x=6, y=401, height=90)

base.mainloop()